# Tahap 1 — Membangun Case Base

Tujuan: mengumpulkan (scraping) dan menyiapkan (cleaning) corpus putusan yang bersih.

**Domain**: Pidana Khusus — K/Pid (Kasasi Pidana)

**Struktur folder proyek:**
```
CBR_Project/
├── data/
│   ├── pdf/          ← letakkan file PDF putusan di sini
│   ├── raws/         ← output .txt hasil ekstraksi
│   ├── processed/    ← output cases.csv, bow_features.csv, dll
│   ├── eval/         ← queries.json, retrieval_metrics.csv
│   └── results/      ← predictions.csv
├── notebooks/        ← notebook ini
└── logs/             ← cleaning.log
```

## Persiapan

In [9]:
import fitz  
import os
import re
from datetime import datetime
import pandas as pd

In [10]:
# PATH KONFIGURASI

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

PDF_FOLDER       = os.path.join(BASE_DIR, "data", "pdf")
OUTPUT_FOLDER    = os.path.join(BASE_DIR, "data", "raws")
LOG_FOLDER       = os.path.join(BASE_DIR, "logs")
PROCESSED_FOLDER = os.path.join(BASE_DIR, "data", "processed")

for folder in [PDF_FOLDER, OUTPUT_FOLDER, LOG_FOLDER, PROCESSED_FOLDER]:
    os.makedirs(folder, exist_ok=True)

LOG_PATH = os.path.join(LOG_FOLDER, "cleaning.log")

print("Base dir :", BASE_DIR)
print("PDF      :", PDF_FOLDER)
print("Raws     :", OUTPUT_FOLDER)
print("Processed:", PROCESSED_FOLDER)
print("Log      :", LOG_PATH)

Base dir : c:\Users\Rani\Downloads\CBR_Project\CBR_Project
PDF      : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\pdf
Raws     : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\raws
Processed: c:\Users\Rani\Downloads\CBR_Project\CBR_Project\data\processed
Log      : c:\Users\Rani\Downloads\CBR_Project\CBR_Project\logs\cleaning.log


### Cek data di folder PDF

In [11]:
files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
print("Jumlah PDF:", len(files))
for f in files[:5]:
    print(" ", f)

Jumlah PDF: 33
  putusan_182_k_pid_2026_20260622123409.pdf
  putusan_205_k_pid_2026_20260622122656.pdf
  putusan_219_k_pid_2026_20260622121831.pdf
  putusan_246_k_pid_2026_20260622134139.pdf
  putusan_254_k_pid_2026_20260622124122.pdf


### Ekstraksi teks dari PDF

In [12]:
def extract_pdf_text(pdf_path: str) -> str:
    """Ekstrak semua teks dari file PDF menggunakan PyMuPDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

## Cleaning
- Hapus header (Direktori Putusan MA RI)
- Hapus nomor halaman
- Hapus footer/disclaimer
- Hapus watermark OCR
- Lowercase
- Normalisasi spasi

In [13]:
def clean_text(text: str) -> str:
    """Bersihkan teks putusan dari header, footer, watermark, dan noise."""
    text = text.lower()

    # 1. Hapus blok header tiap halaman PDF
    text = re.sub(
        r"direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia\s*"
        r"putusan\s*\.?\s*mahkamahagung\s*\.?\s*go\s*\.?\s*id",
        " ", text, flags=re.IGNORECASE
    )
    text = re.sub(
        r"direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia",
        " ", text, flags=re.IGNORECASE
    )
    text = re.sub(
        r"putusan\s*\.?\s*mahkamahagung\s*\.?\s*go\s*\.?\s*id",
        " ", text, flags=re.IGNORECASE
    )

    # 2. Hapus "halaman X dari Y halaman putusan nomor ..."
    text = re.sub(
        r"halaman\s+\d+\s+dari\s+\d+\s+halaman\s+putusan\s+nomor\s+[\w/]+",
        " ", text, flags=re.IGNORECASE
    )

    # 3. Hapus footer disclaimer
    text = re.sub(
        r"disclaimer\s+kepaniteraan.*?(?=\n\s*\n|\Z)",
        " ", text, flags=re.IGNORECASE | re.DOTALL
    )

    # 4. Hapus watermark OCR rusak
    ocr_artifacts = [
        r"^hkama\w*\s*$",
        r"^ahkamah\s+agung\s+repub\w*\s*$",
        r"^ahkamah\s+agung\s+republik\s+\w*\s*$",
        r"^mah\s+agung\s+republik\s+\w*\s*$",
        r"^blik\s+indonesi\w*\s*$",
        r"^a\s+direktori\s+putusan\s*$",
        r"^hkam\w*\s*$",
        r"^ahkam\w*\s*$",
        r"^ma\s*h\s+agung\s*$",
        r"k/pid/2026",
    ]
    for p in ocr_artifacts:
        text = re.sub(p, " ", text, flags=re.IGNORECASE | re.MULTILINE)

    # 5. Hapus baris yang hanya berisi "halaman X"
    text = re.sub(
        r"^\s*halaman\s+\d+\s*$",
        " ", text, flags=re.IGNORECASE | re.MULTILINE
    )

    # 6. Normalisasi whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

### Proses semua PDF → simpan ke .txt (tanpa log)

In [14]:
pdf_files = sorted([f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")])
print("Jumlah PDF:", len(pdf_files))

for i, pdf_file in enumerate(pdf_files, start=1):
    pdf_path = os.path.join(PDF_FOLDER, pdf_file)
    text     = extract_pdf_text(pdf_path)
    clean    = clean_text(text)

    output_file = os.path.join(OUTPUT_FOLDER, f"case_{i:03d}.txt")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(clean)

    print(f"Berhasil: {pdf_file} -> case_{i:03d}.txt")

Jumlah PDF: 33
Berhasil: putusan_182_k_pid_2026_20260622123409.pdf -> case_001.txt
Berhasil: putusan_205_k_pid_2026_20260622122656.pdf -> case_002.txt
Berhasil: putusan_219_k_pid_2026_20260622121831.pdf -> case_003.txt
Berhasil: putusan_246_k_pid_2026_20260622134139.pdf -> case_004.txt
Berhasil: putusan_254_k_pid_2026_20260622124122.pdf -> case_005.txt
Berhasil: putusan_262_k_pid_2026_20260622122855.pdf -> case_006.txt
Berhasil: putusan_270_k_pid_2026_20260622135129.pdf -> case_007.txt
Berhasil: putusan_273_k_pid_2026_20260622122041.pdf -> case_008.txt
Berhasil: putusan_278_k_pid_2026_20260622121917.pdf -> case_009.txt
Berhasil: putusan_308_k_pid_2026_20260622115559.pdf -> case_010.txt
Berhasil: putusan_310_k_pid_2026_20260622122017.pdf -> case_011.txt
Berhasil: putusan_313_k_pid_2026_20260623205950.pdf -> case_012.txt
Berhasil: putusan_362_k_pid_2026_20260622123946.pdf -> case_013.txt
Berhasil: putusan_366_k_pid_2026_20260622123847.pdf -> case_014.txt
Berhasil: putusan_378_k_pid_2026_

### Proses semua PDF + catat log

In [15]:
pdf_files = sorted([f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")])

# Kosongkan log lama
with open(LOG_PATH, "w", encoding="utf-8") as log:
    log.write("=== CLEANING LOG ===\n\n")

for i, pdf_file in enumerate(pdf_files, start=1):
    pdf_path = os.path.join(PDF_FOLDER, pdf_file)
    text     = extract_pdf_text(pdf_path)

    original_chars = len(text)
    original_words = len(text.split())

    clean = clean_text(text)

    cleaned_chars = len(clean)
    cleaned_words = len(clean.split())
    retention     = (cleaned_chars / original_chars * 100) if original_chars > 0 else 0

    output_file = os.path.join(OUTPUT_FOLDER, f"case_{i:03d}.txt")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(clean)

    with open(LOG_PATH, "a", encoding="utf-8") as log:
        log.write(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}]\n")
        log.write(f"Input File     : {pdf_file}\n")
        log.write(f"Output File    : case_{i:03d}.txt\n")
        log.write(f"Original Chars : {original_chars}\n")
        log.write(f"Cleaned Chars  : {cleaned_chars}\n")
        log.write(f"Original Words : {original_words}\n")
        log.write(f"Cleaned Words  : {cleaned_words}\n")
        log.write(f"Retention      : {retention:.2f}%\n")
        log.write("Actions        : remove watermark, remove header/footer, remove page number, normalize whitespace\n")
        log.write(f"Status         : SUCCESS\n")
        log.write("-" * 70 + "\n")

    print(f"[{i}/{len(pdf_files)}] Berhasil: {pdf_file} -> case_{i:03d}.txt ({retention:.2f}%)")

print("\nSelesai!")
print(f"TXT disimpan di : {OUTPUT_FOLDER}")
print(f"Log disimpan di : {LOG_PATH}")

[1/33] Berhasil: putusan_182_k_pid_2026_20260622123409.pdf -> case_001.txt (62.53%)
[2/33] Berhasil: putusan_205_k_pid_2026_20260622122656.pdf -> case_002.txt (61.37%)
[3/33] Berhasil: putusan_219_k_pid_2026_20260622121831.pdf -> case_003.txt (63.39%)
[4/33] Berhasil: putusan_246_k_pid_2026_20260622134139.pdf -> case_004.txt (63.23%)
[5/33] Berhasil: putusan_254_k_pid_2026_20260622124122.pdf -> case_005.txt (63.08%)
[6/33] Berhasil: putusan_262_k_pid_2026_20260622122855.pdf -> case_006.txt (62.20%)
[7/33] Berhasil: putusan_270_k_pid_2026_20260622135129.pdf -> case_007.txt (61.57%)
[8/33] Berhasil: putusan_273_k_pid_2026_20260622122041.pdf -> case_008.txt (63.59%)
[9/33] Berhasil: putusan_278_k_pid_2026_20260622121917.pdf -> case_009.txt (62.91%)
[10/33] Berhasil: putusan_308_k_pid_2026_20260622115559.pdf -> case_010.txt (60.61%)
[11/33] Berhasil: putusan_310_k_pid_2026_20260622122017.pdf -> case_011.txt (63.21%)
[12/33] Berhasil: putusan_313_k_pid_2026_20260623205950.pdf -> case_012.tx

### Validasi — cek keutuhan teks (minimal 80%)

In [16]:
txt_files = sorted([f for f in os.listdir(OUTPUT_FOLDER) if f.endswith(".txt")])
print(f"Jumlah file .txt: {len(txt_files)}")

gagal = []
for f in txt_files:
    path = os.path.join(OUTPUT_FOLDER, f)
    with open(path, "r", encoding="utf-8") as fp:
        content = fp.read()
    if len(content.strip()) < 100:  # threshold minimal
        gagal.append(f)

if gagal:
    print("File dengan isi terlalu pendek:", gagal)
else:
    print("Semua file terlihat valid.")

Jumlah file .txt: 33


Semua file terlihat valid.
